# Projet Covid19 : Préprocessing - Suppression des doublons et outliers

## 1. Présentation du dataset

Ce projet utilise le dataset COVID-19 Radiography Dataset : https://www.kaggle.com/datasets/tawsifurrahman/covid19-radiography-database  
Il contient des radiographies pulmonaires réparties en quatre classes :

- COVID
- Normal
- Lung Opacity
- Viral Pneumonia

Chaque classe contient un dossier `images` avec les radiographies et un dossier `masks` avec les masques pulmonaires associés.

## 2. Environnement

In [1]:
#Vérification de l'environnement de travail

import sys
print(sys.executable)

C:\Users\n_a_e\anaconda3\envs\covid19_env\python.exe


In [8]:
# Chargement des bibliothèques

from pathlib import Path
import pandas as pd

In [3]:
# Chemins d'accés

data_path = Path("../../../COVID-19_Radiography_Dataset")
classes = ["COVID", "Normal", "Lung_Opacity", "Viral Pneumonia"]

print("Chemin absolu :", data_path.resolve())
print("Dossier trouvé :", data_path.exists())

Chemin absolu : C:\Users\n_a_e\Documents\DataScientest\Data Scientist\Projet COVID\COVID-19_Radiography_Dataset
Dossier trouvé : True


## 3. Identification des doublons et outliers

In [4]:
doublons = {
    "COVID": [
        140, 591, 158, 160, 229, 21, 376, 403, 224, 232, 234, 237, 1261, 253, 572,
        1299, 761, 599, 52, 729, 1296, 214, 517, 1078, 1111, 594, 1138, 420, 344,
        602, 227, 532, 624, 768, 758, 169, 724, 1126, 1335, 483, 235, 548, 100,
        178, 181, 190, 525, 633, 372, 766, 1096, 520, 247, 567, 230, 540, 752,
        250, 569, 782, 139, 786, 614, 392, 641, 270, 619, 522, 220, 764, 1167,
        744, 492, 487, 529, 731, 514, 316, 1376, 1373, 792, 1087, 340, 1428, 280,
        419, 848, 1359, 494, 1314, 1443, 497, 1320, 755, 1332, 738, 1326, 355,
        1431, 321, 1123, 1416, 1194, 1188, 428, 463, 1244, 1437, 846, 1440, 595,
        1099, 538, 541, 951, 535, 543, 644, 469, 684, 486, 1452, 794, 1350, 820,
        1356, 873, 1362, 1469, 1264, 1267, 1276, 1279, 1285, 1288, 609, 1291,
        1466, 1353, 695, 1379, 1341, 1407, 1338, 1434, 1344, 1347, 1389, 1392,
        1395, 1425, 1455, 898, 804, 1458, 1511, 1404, 1526, 1410, 1472, 770,
        1323, 1228, 1574, 1367, 1370, 1422, 1413, 1419, 1505, 1386, 1446, 1449,
        1461, 1520, 1553, 1448, 2085, 2418, 2420, 2421, 2423, 2432, 2459, 2493,
        2667, 2678, 2716, 2717, 2718, 2719, 2773, 2774, 2805, 2808, 2809, 2810,
        2811, 2812, 2813, 2823, 2942, 2946, 3141, 3142, 3143, 3144, 3208, 3212,
        3223, 3299, 3374, 3375, 3376, 3403, 3427, 3429, 3489, 3516, 3548, 3574,
        3591, 3599
    ],
    "Normal": [818],
    "Lung_Opacity": [],
    "Viral Pneumonia": [75, 118, 250, 295, 596, 954, 1053]
}

In [6]:
print("Nombre de doublons à supprimer :")
for classe, liste in doublons.items():
    print(f"{classe:<18}: {len(liste)}")

print()

Nombre de doublons à supprimer :
COVID             : 223
Normal            : 1
Lung_Opacity      : 0
Viral Pneumonia   : 7



In [5]:
outliers = {
    "COVID": [
        324, 500, 585, 1141, 1401, 1413, 1553, 1703, 1730,
        1740, 1742, 2471, 2492, 2493, 2647, 2688, 2750,
        3480, 3584, 3615, 3342, 1179, 1876
    ],

    "Normal": [
        1485, 1530, 1591, 1834, 2287, 2857, 3068, 3117,
        3251, 3336, 3341, 3344, 3777, 3795, 3872, 3906,
        3921, 4355, 4820, 4869, 5188, 5207, 5224, 5400,
        5411, 5422, 5545, 5674, 6128, 6209, 6811, 7213,
        7383, 7501, 7516, 7619, 7634, 7764, 7826, 7859,
        7887, 7932, 8217, 8295, 8694, 8729, 8749, 9147,
        9381, 9390, 9997, 10146, 4213, 4206, 2001
    ],

    "Lung_Opacity": [
        12, 66, 191, 306, 588, 889, 1212, 1657, 2282,
        2617, 2731, 2959, 2982, 3395, 3560, 3570,
        3608, 4351, 4540, 4776, 4791, 4871, 4872, 5013
    ],

    "Viral Pneumonia": []
}

In [7]:
print("Nombre d'outliers à supprimer :")
for classe, liste in outliers.items():
    print(f"{classe:<18}: {len(liste)}")

Nombre d'outliers à supprimer :
COVID             : 23
Normal            : 55
Lung_Opacity      : 24
Viral Pneumonia   : 0


## 4. Définition des focntions

In [9]:
def supprimer_fichiers(data_path, fichiers_a_supprimer, dry_run=False):

    rapport = []

    for classe, numeros in fichiers_a_supprimer.items():

        for num in numeros:

            filename = f"{classe}-{num}.png"

            image_path = data_path / classe / "images" / filename
            mask_path = data_path / classe / "masks" / filename

            for file_path, type_fichier in [
                (image_path, "image"),
                (mask_path, "mask")
            ]:

                existe = file_path.exists()

                if existe and not dry_run:
                    file_path.unlink()

                rapport.append({
                    "classe": classe,
                    "numero": num,
                    "fichier": filename,
                    "type": type_fichier,
                    "existe": existe,
                    "supprime": existe and not dry_run
                })

    return pd.DataFrame(rapport)

In [10]:
def compter_dataset(data_path):

    classes = ["COVID", "Normal", "Lung_Opacity", "Viral Pneumonia"]

    resultat = []

    for classe in classes:

        images = list((data_path/classe/"images").glob("*.png"))
        masks  = list((data_path/classe/"masks").glob("*.png"))

        resultat.append({
            "classe": classe,
            "nb_images": len(images),
            "nb_masks": len(masks)
        })

    return pd.DataFrame(resultat)

## 5. Suppression des doublons

In [11]:
print("État initial")
compter_dataset(data_path)

État initial


,classe,nb_images,nb_masks
0,COVID,3616,3616
1,Normal,10192,10192
2,Lung_Opacity,6012,6012
3,Viral Pneumonia,1345,1345


In [12]:
rapport_doublons = supprimer_fichiers(
    data_path,
    doublons,
    dry_run=False
)

print("État après suppression des doublons")
compter_dataset(data_path)

État après suppression des doublons


,classe,nb_images,nb_masks
0,COVID,3393,3393
1,Normal,10191,10191
2,Lung_Opacity,6012,6012
3,Viral Pneumonia,1338,1338


## 6. Suppression des outliers

In [13]:
rapport_outliers = supprimer_fichiers(
    data_path,
    outliers,
    dry_run=False
)

print("État final")
compter_dataset(data_path)

État final


,classe,nb_images,nb_masks
0,COVID,3373,3373
1,Normal,10136,10136
2,Lung_Opacity,5988,5988
3,Viral Pneumonia,1338,1338
